In [ ]:
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")

In [ ]:
# ERRO CORRIGIDO: estava escrito "faca o path" (texto solto, não é código). Agora é uma variável path com o caminho do arquivo.
path = r"Tabela_11_Domicilios_sem_presenca_de_conjuge_e_com_filhos.xlsx" 

# Sexo da pessoa responsável pelo domicílio

In [ ]:
#Lé todas as abas do arquivo
# ERRO CORRIGIDO: estava "parametro para sheet name=None". O parâmetro certo é sheet_name=None (lê todas as abas).
todas = pd.read_excel(path, sheet_name=None, header=None)
print(todas.keys())

In [ ]:
# ERRO CORRIGIDO: estava "pd.COMO LER(". A função correta é pd.read_excel().
bruto = pd.read_excel(path, sheet_name="Tabela base do SIDRA 9882", header=None)

In [ ]:
bruto

In [ ]:
# ERRO CORRIGIDO: estava "bruto.PARAMETRO PARA CORTAR A TABELA POR INDICE[5:5608]". Cortar por posição usa .iloc[].
indicador_2 = bruto.iloc[5:5608].copy()

In [ ]:
indicador_2

In [ ]:
# Preenchendo os nomes das colunas
indicador_2.columns = [
    "regiao",
    "total",
    "homens",
    "mulheres",
]


In [ ]:
# ERRO CORRIGIDO: estava "indicador_2.PARAMETRO PARA RESETAR O INDEX(drop=True)". O método é .reset_index(drop=True).
indicador_2 = indicador_2.reset_index(drop=True)

# valores são número, não texto
for col in indicador_2.columns[1:]:
    indicador_2[col] = pd.to_numeric(indicador_2[col], errors="coerce")

indicador_2

In [ ]:
# ERRO CORRIGIDO: estava "indicador_2.QUAL PARAMETRO USO PARA DESCREVER()". O método é .describe().
indicador_2.describe()

In [ ]:
indicador_2.describe().round(2)

EXCLUIR AS CIDADES, REPAREM NO PADRÃO, ELAS TEM O () QUE INDICAM O UF

In [ ]:
# ERRO CORRIGIDO: faltava a negação. Estava "...str.contains(...) INSIRA UMA NEGAÇÃO". O ~ inverte o filtro: mantém só quem NÃO tem "(" (tira os municípios).
indicador_2 = indicador_2[~indicador_2["regiao"].str.contains(r"\(", na=False)].copy()
indicador_2 

# Trazer a Tabela 1.1.1 (horas de trabalho doméstico) e juntar com indicador_2

In [ ]:
# Mesma limpeza da Aula 2: linhas 8–40, nomes de coluna, horas como número
path_horas = r"PATH TABELA AULA 2"
bruto_1 = pd.read_excel(path_horas, engine="xlrd", header=None)

indicador_1 = bruto_1.iloc[8:41].copy()
indicador_1.columns = [
    "uf_regiao",
    "total",
    "total_branca",
    "total_preta_parda",
    "homem_branca",
    "homem_preta_parda",
    "mulher_branca",
    "mulher_preta_parda",
]
indicador_1 = indicador_1.reset_index(drop=True)
for col in indicador_1.columns[1:]:
    indicador_1[col] = pd.to_numeric(indicador_1[col], errors="coerce")

indicador_1

In [ ]:
# Left join: fica tudo de indicador_2; puxa as horas da 1.1.1 quando o nome bate
# suffixes: as duas tabelas têm coluna "total" — domicílios vs horas
indicador1e2 = indicador_2.merge(
    indicador_1,
    how="left",
    left_on="regiao",
    right_on="uf_regiao",
    suffixes=("_domicilios", "_horas"),
)
indicador1e2 = indicador1e2.drop(columns=["uf_regiao"])


In [ ]:
indicador1e2 = indicador1e2.rename(columns={
    "homens": "resp_domicilio_homens",
    "mulheres": "resp_domicilio_mulheres",
})
indicador1e2

In [ ]:
# % de domicílios com responsável mulher  |  horas médias das mulheres (média das duas raças)
indicador1e2["pct_resp_mulheres"] = (
    indicador1e2["resp_domicilio_mulheres"] / indicador1e2["total_domicilios"] * 100
)
indicador1e2["horas_mulher"] = (
    indicador1e2["mulher_branca"] + indicador1e2["mulher_preta_parda"]
) / 2

indicador1e2[
    ["regiao", "pct_resp_mulheres", "horas_mulher", "resp_domicilio_mulheres", "total_domicilios"]
].sort_values("pct_resp_mulheres", ascending=False)

pct_resp_mulheres = em cada 100 desses domicílios, quantos têm mulher responsável.
horas_mulher = média semanal de horas das mulheres (média simples de branca e preta/parda na 1.1.1).
Como ler os dois juntos. Nordeste concentra os dois: Sergipe ~90% de responsáveis mulheres e ~24 h; Alagoas ~89% e ~25 h (mais horas). Norte tem menos lares com responsável mulher (Amazonas ~82%) e menos horas (Amapá, Acre, Roraima ~16 h). DF é o contraste: muitas responsáveis mulheres (~87%) e menos horas (~18,5 h).

In [ ]:
# Neste recorte (sem cônjuge + com filhos): mulheres são maioria?
# ERRO CORRIGIDO: estava "COLOQUE AQUI O SINAL DE MAIOR". O sinal é > (mulheres maior que homens).
indicador_2["mais_mulheres"] = indicador_2["mulheres"] > indicador_2["homens"]

# ERRO CORRIGIDO: estava "COLOQUE AQUI O SINAL QUE IGUALA ATRIBUTOS". Comparação de igualdade é == (o = sozinho é atribuição).
brasil = indicador_2.loc[indicador_2["regiao"] == "Brasil"].iloc[0]
print("Brasil — domicílios sem cônjuge e com filhos (Censo 2022)")
print(f"  mulheres: {int(brasil.mulheres):,}  ({brasil.mulheres/brasil.total*100:.1f}%)")
print(f"  homens:   {int(brasil.homens):,}  ({brasil.homens/brasil.total*100:.1f}%)")
print(f"  {brasil.mulheres/brasil.homens:.1f} mulheres para cada homem responsável")
print()
print("Em todas as UFs/regiões mulheres > homens?", indicador_2["mais_mulheres"].all())

indicador_2[["regiao", "homens", "mulheres", "mais_mulheres"]]

## Gráficos com matplotlib e seaborn

Visualizamos o recorte **domicílios sem cônjuge e com filhos** e as **horas de afazeres** da Tabela 1.1.1.

In [ ]:
regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
ufs = indicador1e2[~indicador1e2["regiao"].isin(["Brasil"] + regioes)].copy()

# Barras: % de domicílios com responsável mulher
# ERRO CORRIGIDO: estava "PARAMETRO PARA DESENHAR A FIGURA" -> fig; e "PARAMETRO PARA O TAMANHO DA FIGURA" -> figsize. (fig, ax = plt.subplots(figsize=...))
fig, ax = plt.subplots(figsize=(9, 8))
ordem = ufs.sort_values("pct_resp_mulheres", ascending=False)
sns.barplot(data=ordem, y="regiao", x="pct_resp_mulheres", ax=ax, color="teal")
ax.set_title("% de domicílios com responsável mulher (sem cônjuge + filhos)")
# ERRO CORRIGIDO: estava "ax.PARAMETRO PARA A LABEL DO EIXO X(...)". O método é ax.set_xlabel(...).
ax.set_xlabel("% com responsável mulher")
ax.set_ylabel("UF")
plt.tight_layout()
# ERRO CORRIGIDO: estava "plt.PARAMETRO PARA MOSTRAR()". O método é plt.show().
plt.show()

In [ ]:
# Barras agrupadas: responsáveis homens x mulheres por grande região
reg = indicador_2[indicador_2["regiao"].isin(regioes)].copy()
# ERRO CORRIGIDO: estava "QUAL QUE DERRETE?(". O método que "derrete" a tabela (largo -> longo) é .melt().
reg_long = reg.melt(
    id_vars="regiao",
    value_vars=["homens", "mulheres"],
    var_name="sexo",
    value_name="domicilios",
)

# ERRO CORRIGIDO: estava "PARAMETRO PARA DESENHAR A FIGURA" -> fig; e "PARAMETRO PARA O TAMANHO DA FIGURA" -> figsize. (fig, ax = plt.subplots(figsize=...))
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=reg_long, x="regiao", y="domicilios", hue="sexo", ax=ax)
ax.set_title("Domicílios sem cônjuge e com filhos, por região e sexo do responsável")
# ERRO CORRIGIDO: estava "ax.PARAMETRO PARA A LABEL DO EIXO X(...)". O método é ax.set_xlabel(...).
ax.set_xlabel("Região")
ax.set_ylabel("Número de domicílios")
plt.tight_layout()
# ERRO CORRIGIDO: estava "plt.PARAMETRO PARA MOSTRAR()". O método é plt.show().
plt.show()

In [ ]:
# Histograma (frequência) e box plot das horas das mulheres
# ERRO CORRIGIDO: estava "PARAMETRO PARA DESENHAR A FIGURA" -> fig; e "PARAMETRO PARA O TAMANHO DA FIGURA" -> figsize. (fig, ax = plt.subplots(figsize=...))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(ufs["horas_mulher"], bins=8, kde=True, ax=axes[0])
axes[0].set_title("Frequência das horas médias das mulheres (UFs)")
axes[0].set_xlabel("Horas / semana")
axes[0].set_ylabel("Número de UFs")

sns.boxplot(y=ufs["horas_mulher"], ax=axes[1])
axes[1].set_title("Box plot das horas das mulheres por UF")
axes[1].set_ylabel("Horas / semana")
plt.tight_layout()
# ERRO CORRIGIDO: estava "plt.PARAMETRO PARA MOSTRAR()". O método é plt.show().
plt.show()

In [ ]:
# Box plot: % de responsáveis mulheres por UF
# ERRO CORRIGIDO: estava "PARAMETRO PARA DESENHAR A FIGURA" -> fig; e "PARAMETRO PARA O TAMANHO DA FIGURA" -> figsize. (fig, ax = plt.subplots(figsize=...))
fig, ax = plt.subplots(figsize=(5, 4))
sns.boxplot(y=ufs["pct_resp_mulheres"], ax=ax, color="teal")
ax.set_title("Box plot: % responsável mulher nas UFs")
ax.set_ylabel("%")
plt.tight_layout()
# ERRO CORRIGIDO: estava "plt.PARAMETRO PARA MOSTRAR()". O método é plt.show().
plt.show()

In [ ]:
# Dispersão: será que mais responsáveis mulheres anda junto com mais horas em casa?
# ERRO CORRIGIDO: estava "PARAMETRO PARA DESENHAR A FIGURA" -> fig; e "PARAMETRO PARA O TAMANHO DA FIGURA" -> figsize. (fig, ax = plt.subplots(figsize=...))
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=ufs, x="pct_resp_mulheres", y="horas_mulher", ax=ax, s=80)
for _, row in ufs.iterrows():
    ax.annotate(row["regiao"], (row["pct_resp_mulheres"], row["horas_mulher"]), fontsize=7)
ax.set_title("Responsável mulher (%) × horas de afazeres das mulheres")
# ERRO CORRIGIDO: estava "ax.PARAMETRO PARA A LABEL DO EIXO X(...)". O método é ax.set_xlabel(...).
ax.set_xlabel("% domicílios com responsável mulher")
ax.set_ylabel("Horas / semana (mulheres)")
plt.tight_layout()
# ERRO CORRIGIDO: estava "plt.PARAMETRO PARA MOSTRAR()". O método é plt.show().
plt.show()

In [ ]:
# Frequência: neste recorte, mulheres são maioria entre os responsáveis?
ufs["maioria"] = "Mais mulheres"
ufs.loc[ufs["resp_domicilio_mulheres"] <= ufs["resp_domicilio_homens"], "maioria"] = "Mais homens"

# ERRO CORRIGIDO: estava "PARAMETRO PARA DESENHAR A FIGURA" -> fig; e "PARAMETRO PARA O TAMANHO DA FIGURA" -> figsize. (fig, ax = plt.subplots(figsize=...))
fig,ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=ufs, x="maioria", ax=ax)
ax.set_title("Frequência de UFs segundo o sexo da maioria dos responsáveis")
# ERRO CORRIGIDO: estava "ax.PARAMETRO PARA A LABEL DO EIXO X(...)". O método é ax.set_xlabel(...).
ax.set_xlabel("")
ax.set_ylabel("Número de UFs")
plt.tight_layout()
# ERRO CORRIGIDO: estava "plt.PARAMETRO PARA MOSTRAR()". O método é plt.show().
plt.show()